# Task 8: The "Risk Divergence" Analysis
## Block, Inc. vs. PayPal Holdings — 10-K 2023 Risk Factor Comparison

---

### Objective
Identify whether Block's new **AI & Bitcoin** focus is creating **undisclosed operational risks**
compared to PayPal's conservative, efficiency-first approach.
We use **three distinct prompting strategies** and an **LLM-as-Judge** framework to rate each
prompt's analytical depth, then synthesise findings into a final Executive Summary.

### Methodology
| Step | Description |
|------|-------------|
| 1 | Extract & categorise Risk Factors from both 10-K filings |
| 2 | Apply **3 distinct prompting techniques** to surface subtle risks |
| 3 | Use an **LLM-as-Judge** rubric to rate each prompt's output |
| 4 | Visualise divergence across risk categories |
| 5 | Write a 200-word Executive Summary |

---

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Dependency Installation
# Uses sys.executable to ensure packages install into the SAME Python kernel
# that is running this notebook, avoiding the 'externally managed' env error.
# ─────────────────────────────────────────────────────────────────────────────
import sys
import subprocess

packages = ['groq', 'pandas', 'numpy', 'matplotlib', 'seaborn']
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages,
    check=True
)
print('✓ All dependencies installed successfully')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Import All Libraries
# Centralising imports here prevents NameErrors in later cells.
# ─────────────────────────────────────────────────────────────────────────────
import os
import re
import json
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from groq import Groq          # Groq client — no openai dependency needed

# Matplotlib aesthetic settings
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

print('✓ All imports successful')

---
## Section 1 — Risk Factor Taxonomy

We manually catalogue **risk-factor headlines** extracted from the Block 10-K (FY 2022/2023)
and PayPal 10-K (FY 2023) into **six comparable buckets**.

Two scores are assigned per risk item:
- **Severity Score (1–5):** importance / impact magnitude based on filing language intensity
- **Disclosure Explicitness (1–5):** how quantified / detailed the disclosure is (1 = vague; 5 = multi-paragraph with numbers)

A high Severity + low Explicitness combination indicates a potential **undisclosed / underweighted risk**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Risk Factor Catalogue
# Source: Block 10-K 2023 (SEC EDGAR) and PayPal 10-K 2023 (SEC EDGAR)
# Scores are analyst estimates based on filing language intensity & prominence
# ─────────────────────────────────────────────────────────────────────────────

risk_data = {
    'Category': [
        # ── Regulatory & Compliance ───────────────────────────────────────────
        'Regulatory & Compliance',
        'Regulatory & Compliance',
        'Regulatory & Compliance',
        'Regulatory & Compliance',
        'Regulatory & Compliance',
        'Regulatory & Compliance',
        # ── Cryptocurrency / Digital Asset ────────────────────────────────────
        'Cryptocurrency / Digital Assets',
        'Cryptocurrency / Digital Assets',
        'Cryptocurrency / Digital Assets',
        'Cryptocurrency / Digital Assets',
        # ── Cybersecurity / Data Privacy ──────────────────────────────────────
        'Cybersecurity & Data Privacy',
        'Cybersecurity & Data Privacy',
        'Cybersecurity & Data Privacy',
        'Cybersecurity & Data Privacy',
        # ── Technology & Innovation ───────────────────────────────────────────
        'Technology & Innovation',
        'Technology & Innovation',
        'Technology & Innovation',
        'Technology & Innovation',
        # ── Financial / Market ────────────────────────────────────────────────
        'Financial & Market',
        'Financial & Market',
        'Financial & Market',
        'Financial & Market',
        # ── Operational / Strategic ───────────────────────────────────────────
        'Operational & Strategic',
        'Operational & Strategic',
        'Operational & Strategic',
        'Operational & Strategic',
    ],
    'Company': [
        'Block', 'Block', 'PayPal', 'PayPal', 'Block', 'PayPal',
        'Block', 'Block', 'PayPal', 'PayPal',
        'Block', 'PayPal', 'Block', 'PayPal',
        'Block', 'Block', 'PayPal', 'PayPal',
        'Block', 'Block', 'PayPal', 'PayPal',
        'Block', 'Block', 'PayPal', 'PayPal',
    ],
    'Risk Factor': [
        # ── Regulatory & Compliance ───────────────────────────────────────────
        'Money transmitter licensing across 50+ jurisdictions',
        'Square Financial Services (Utah industrial bank) — Dodd-Frank oversight',
        'Extensive money transmitter licenses in ~200 markets',
        'Luxembourg credit institution (PayPal Europe) — ECB PISA framework',
        'Broker-dealer regulation of Cash App Investing (FINRA)',
        'Cryptocurrency regulation uncertainty — NY BitLicense + SEC securities claims',
        # ── Cryptocurrency / Digital Asset ────────────────────────────────────
        'Bitcoin balance sheet holdings — impairment & volatile market prices',
        'Safeguarding customer bitcoin — custodial failure, theft, or loss',
        'Customer crypto via third-party custodian — insurance insufficiency',
        'SEC may classify supported cryptocurrencies as securities',
        # ── Cybersecurity / Data Privacy ──────────────────────────────────────
        'Data breach / improper access to sensitive seller & customer data',
        'Cyberattacks, ransomware, DDoS — high-profile target risk',
        'System failures impacting payment processing uptime',
        'TIO Network breach precedent (1.6M customers compromised)',
        # ── Technology & Innovation ───────────────────────────────────────────
        'TIDAL integration complexity and license payment estimation',
        'TBD (blockchain/Web3) and Spiral (bitcoin hardware) — nascent stage risks',
        'Inability to keep pace with AI/ML and payment innovations',
        'New technology adoption dependence on third-party platform access',
        # ── Financial / Market ────────────────────────────────────────────────
        'Net loss of $540.7M (2022); accumulated deficit $568.7M',
        'Afterpay BNPL integration risk and BNPL regulatory scrutiny',
        'Revenue concentration — TPV $1.36T but under margin pressure',
        'Foreign exchange rate exposure across global operations',
        # ── Operational / Strategic ───────────────────────────────────────────
        'Dependence on key management and talent retention',
        'Ecosystem over-extension: Square + Cash App + TIDAL + TBD + Spiral',
        'Dependence on payment card networks and acquiring processors',
        'Business continuity obligations — regulator-mandated DR testing',
    ],
    # Severity: 1 = minor mention | 5 = primary, extensively described risk
    'Severity_Score': [
        4, 5, 4, 4, 3, 5,      # Regulatory
        5, 5, 3, 4,             # Crypto
        4, 5, 4, 3,             # Cybersecurity
        3, 5, 4, 3,             # Technology
        5, 4, 3, 3,             # Financial
        3, 5, 4, 3,             # Operational
    ],
    # Explicitness: 1 = vague | 5 = detailed, quantified, multi-paragraph
    'Disclosure_Explicitness': [
        3, 5, 4, 3, 3, 4,      # Regulatory
        5, 4, 3, 3,             # Crypto
        4, 5, 3, 4,             # Cybersecurity
        3, 3, 4, 3,             # Technology
        5, 4, 3, 3,             # Financial
        3, 3, 4, 4,             # Operational
    ],
}

df = pd.DataFrame(risk_data)

print(f'Total risk factors catalogued : {len(df)}')
print(f'  Block  : {(df.Company=="Block").sum()} items')
print(f'  PayPal : {(df.Company=="PayPal").sum()} items')
print()
df.head(10)

---
## Section 2 — Prompting Techniques for Risk Divergence Detection

We use **three distinct prompting strategies** via the Groq-hosted LLM API,
each designed to surface a different angle of hidden risk:

| # | Technique | Goal |
|---|-----------|------|
| **P1** | *Zero-Shot Direct Comparison* | Baseline: compare risk factors head-to-head |
| **P2** | *Chain-of-Thought with Semantic Contrast* | Step-by-step omission analysis — what is NOT said |
| **P3** | *Red-Team Adversarial Lens* | Short-seller bear case: find every concealed / underweighted risk |

All three prompts receive the same **shared context** extracted from the 10-K filings.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Shared Context Block
# Distilled from Block 10-K 2023 and PayPal 10-K 2023 (SEC EDGAR filings).
# This context is injected into every prompt to ensure a consistent knowledge base.
# ─────────────────────────────────────────────────────────────────────────────

SHARED_CONTEXT = """
COMPANY PROFILES (from 10-K filings, fiscal year 2022/2023):

=== BLOCK, INC. ===
Strategy: Aggressive ecosystem expansion — Square (merchant POS), Cash App (consumer finance),
TIDAL (music), TBD (Bitcoin/Web3), Spiral (Bitcoin hardware/software).
Financials: Net loss $540.7M; accumulated deficit $568.7M; holds Bitcoin on balance sheet.
Key risk disclosures:
- Bitcoin holdings subject to impairment and volatile market prices
- Custodial obligation to safeguard customer bitcoin — theft/loss risk explicitly mentioned
- Square Financial Services (Utah industrial bank) under Dodd-Frank — deposit insurance exposure
- Cash App Investing regulated as broker-dealer (FINRA)
- TIDAL license payment estimation described as inherently uncertain
- TBD (blockchain) and Spiral described as 'nascent stage' with no clear revenue model
- Afterpay BNPL integration — regulatory scrutiny of BNPL sector
- Extensive use of dual-class stock; founder control retained

=== PAYPAL HOLDINGS ===
Strategy: Stable incumbent — efficiency-first, pruning non-core assets, focusing on core
checkout (PayPal, Venmo, Braintree, Zettle, Honey, Paidy).
Financials: TPV $1.36T (+9%); 22.3B transactions (+16%); 435M active accounts (+2%).
Key risk disclosures:
- Cyber threats highlighted as primary risk (dedicated leading section); TIO breach precedent cited
- Luxembourg credit institution (PayPal Europe) subject to ECB PISA regulatory framework
- Customer crypto via third-party custodian — insurance insufficiency noted but vague
- Regulatory complexity across ~200 markets (money transmitter licenses)
- AI/ML pace-of-change risk acknowledged but framed as opportunity-risk balance
- BNPL not explicitly cited as a systemic risk (unlike Block)
- No balance-sheet crypto holdings
"""

print('✓ Shared context prepared.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — LLM Client Initialisation (Groq)
#
# We use the Groq SDK directly (no openai dependency needed).
# The model 'llama-3.3-70b-versatile' is a strong open-weight model
# served at near-zero latency on Groq's LPU hardware.
#
# Get a free API key at: https://console.groq.com → API Keys
# ─────────────────────────────────────────────────────────────────────────────

# ⚠️  Replace the key below with your own Groq API key
GROQ_API_KEY = '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'

client = Groq(api_key=GROQ_API_KEY)
MODEL  = 'llama-3.3-70b-versatile'


def call_llm(system_prompt: str, user_prompt: str, max_tokens: int = 1024) -> str:
    """
    Single-turn LLM call via Groq.

    Parameters
    ----------
    system_prompt : str  — Persona / behavioural instruction for the model
    user_prompt   : str  — The actual analysis task
    max_tokens    : int  — Upper limit on response length

    Returns
    -------
    str  — Model's text response
    """
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
    )
    return response.choices[0].message.content


print(f'✓ Groq client ready  |  model = {MODEL}')

### Prompt 1 — Zero-Shot Direct Comparison

**Technique:** The simplest approach — ask the model to compare risk disclosures head-to-head
without any explicit reasoning scaffold.

**Purpose:** Establishes a baseline. If the zero-shot output is already insightful, it reduces
the marginal value of more complex prompting techniques.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Prompt 1: Zero-Shot Direct Comparison
#
# Design rationale:
#   - No chain-of-thought; no adversarial framing
#   - System prompt enforces evidence-based, filing-specific language
#   - Asks for exactly 5 divergence areas to constrain scope
# ─────────────────────────────────────────────────────────────────────────────

P1_SYSTEM = """You are a senior financial analyst specialising in fintech regulatory risk.
Your analysis must be precise, evidence-based, and reference specific 10-K disclosures."""

P1_USER = f"""{SHARED_CONTEXT}

TASK: Compare the Risk Factor disclosures of Block, Inc. and PayPal Holdings.
Identify the top 5 areas where Block's risk profile diverges most significantly from PayPal's,
and explain whether each divergence represents an *undisclosed* or *underweighted* operational risk.
Be concise: one short paragraph per divergence area.
"""

print('Running Prompt 1 — Zero-Shot Direct Comparison...')
p1_response = call_llm(P1_SYSTEM, P1_USER, max_tokens=900)

print('\n' + '='*70)
print('  PROMPT 1 OUTPUT — Zero-Shot Direct Comparison')
print('='*70)
print(p1_response)

### Prompt 2 — Chain-of-Thought with Semantic Contrast (Omission Analysis)

**Technique:** Forces the model to reason *step by step*, explicitly labelling each reasoning stage.
The key innovation is the **semantic contrast**: the model must identify what is present in one
filing but absent (or minimal) in the other — i.e., the *omission* as a signal.

**Purpose:** Surfaces risks that companies strategically downplay or bury.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7 — Prompt 2: Chain-of-Thought + Semantic Contrast (Omission Analysis)
#
# Design rationale:
#   - Five explicit numbered steps force structured reasoning
#   - Steps 1-2 focus on Block's unique disclosures and the motivation behind them
#   - Steps 3-4 flip the lens to find what Block DOESN'T say vs PayPal
#   - Step 5 synthesises into a verdict on undisclosed risk
# ─────────────────────────────────────────────────────────────────────────────

P2_SYSTEM = """You are an expert risk auditor who specialises in detecting *omission bias* in
SEC filings — risks that companies strategically downplay or bury. Think step by step."""

P2_USER = f"""{SHARED_CONTEXT}

TASK — follow these steps explicitly and label each step:

STEP 1 — LIST all material risk categories present in Block's filing but ABSENT or MINIMAL in PayPal's.
STEP 2 — For each item in Step 1, reason about WHY Block might be choosing to disclose it more prominently
         than PayPal (regulatory pressure? materiality threshold? investor relations strategy?).
STEP 3 — LIST all material risk categories present in PayPal's filing but ABSENT or MINIMAL in Block's.
STEP 4 — For each item in Step 3, assess whether Block's *silence* on this risk represents a genuine
         lower exposure or a potentially undisclosed / underweighted risk.
STEP 5 — Conclude with a 3-sentence verdict: Is Block's AI & Bitcoin focus creating UNDISCLOSED
         operational risks beyond what is visible in current 10-K disclosures?
"""

print('Running Prompt 2 — Chain-of-Thought Omission Analysis...')
p2_response = call_llm(P2_SYSTEM, P2_USER, max_tokens=1200)

print('\n' + '='*70)
print('  PROMPT 2 OUTPUT — Chain-of-Thought Omission Analysis')
print('='*70)
print(p2_response)

### Prompt 3 — Red-Team Adversarial Lens (Short-Seller Perspective)

**Technique:** Assigns the model a *hostile persona* — a hedge-fund short-seller whose
job is to build a bear case. This adversarial framing is known to surface risks
that neutral or bullish prompts miss.

**Purpose:** Stress-test disclosures against the worst-case interpretation.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 — Prompt 3: Red-Team Adversarial Lens — Short-Seller Perspective
#
# Design rationale:
#   - Persona: hostile short-seller whose incentive is to find hidden risks
#   - Three structured headings force coverage of specific risk domains
#   - Severity + likelihood scores (1-10) make the output quantitatively comparable
#   - Overall Risk Divergence Score provides a single headline number
# ─────────────────────────────────────────────────────────────────────────────

P3_SYSTEM = """You are a hedge fund short-seller writing a bear-case thesis for Block, Inc.
Your goal is to identify every subtle, underweighted, or potentially concealed risk in Block's
10-K that could trigger a regulatory crackdown, operational failure, or market repricing.
Draw contrasts to PayPal's disclosures to argue that Block is taking on MORE hidden risk.
Be adversarial, specific, and cite disclosure language where possible."""

P3_USER = f"""{SHARED_CONTEXT}

TASK: Write a structured short-seller bear case for Block, Inc. under three headings:

A) REGULATORY POWDER KEG — Regulatory risks Block downplays vs. PayPal's fuller disclosure.
B) BITCOIN CONCENTRATION RISK — Specific undisclosed or under-quantified bitcoin-related risks.
C) ECOSYSTEM OVERREACH — Strategic risks from TIDAL, TBD, Spiral that are vague or unquantified.

Under each heading, score the severity (1–10) and likelihood of materialisation (1–10)
in the next 24 months. End with an OVERALL RISK DIVERGENCE SCORE (1–10) vs PayPal.
"""

print('Running Prompt 3 — Adversarial Red-Team / Short-Seller...')
p3_response = call_llm(P3_SYSTEM, P3_USER, max_tokens=1100)

print('\n' + '='*70)
print('  PROMPT 3 OUTPUT — Adversarial Red-Team (Short-Seller)')
print('='*70)
print(p3_response)

---
## Section 3 — LLM-as-Judge

A **separate LLM call** (same model acting as a neutral evaluator) rates all three prompt outputs
against a standardised rubric on **four criteria**:

| Criterion | What it measures |
|-----------|------------------|
| **Specificity** | Does it cite concrete disclosures, numbers, regulatory frameworks? |
| **Depth** | Does it go beyond surface-level to uncover subtle / hidden risks? |
| **Actionability** | Are findings actionable for an investor or risk manager? |
| **Objectivity** | Is the analysis balanced, or biased by the prompt's framing? |

The judge returns **structured JSON** to enable programmatic downstream analysis.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 — LLM-as-Judge: Rate the Three Prompt Outputs
#
# Key design choices:
#   - Judge is told to return ONLY valid JSON (no prose outside the block)
#   - We strip markdown fences with regex before json.loads to be robust
#   - The judge sees all three responses simultaneously for fair cross-comparison
# ─────────────────────────────────────────────────────────────────────────────

JUDGE_SYSTEM = """You are an impartial analytical judge evaluating the quality of financial risk analyses.
Score each submission on a scale of 1-10 for each criterion. Be rigorous and consistent.
Return your evaluation ONLY as valid JSON — no prose outside the JSON block."""

JUDGE_USER = f"""You have received three analyst responses about risk divergence between Block, Inc.
and PayPal Holdings. Evaluate each on these four criteria:

1. SPECIFICITY   — Does it cite concrete disclosures, numbers, and regulatory frameworks?
2. DEPTH         — Does it go beyond surface-level observations to uncover subtle/hidden risks?
3. ACTIONABILITY — Are the findings actionable for an investor or risk manager?
4. OBJECTIVITY   — Is the analysis balanced, or is it biased by the framing of the prompt?

=== RESPONSE 1 (Zero-Shot Direct Comparison) ===
{p1_response}

=== RESPONSE 2 (Chain-of-Thought Omission Analysis) ===
{p2_response}

=== RESPONSE 3 (Adversarial Red-Team / Short-Seller) ===
{p3_response}

Return JSON in this exact format:
{{
  "prompt_1": {{"specificity": <1-10>, "depth": <1-10>, "actionability": <1-10>, "objectivity": <1-10>, "overall": <1-10>, "comment": "<one sentence>"}},
  "prompt_2": {{"specificity": <1-10>, "depth": <1-10>, "actionability": <1-10>, "objectivity": <1-10>, "overall": <1-10>, "comment": "<one sentence>"}},
  "prompt_3": {{"specificity": <1-10>, "depth": <1-10>, "actionability": <1-10>, "objectivity": <1-10>, "overall": <1-10>, "comment": "<one sentence>"}},
  "best_prompt": "<prompt_1 | prompt_2 | prompt_3>",
  "judge_rationale": "<two sentences explaining the winner>"
}}
"""

print('Running LLM-as-Judge evaluation...')
judge_raw = call_llm(JUDGE_SYSTEM, JUDGE_USER, max_tokens=700)

# Strip potential markdown code fences before parsing
json_text    = re.sub(r'```json|```', '', judge_raw).strip()
judge_scores = json.loads(json_text)

print('\n' + '='*70)
print('  JUDGE EVALUATION — Parsed JSON')
print('='*70)
print(json.dumps(judge_scores, indent=2))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10 — Judge Score Summary Table
# Displays the judge scores as a clean, human-readable DataFrame.
# ─────────────────────────────────────────────────────────────────────────────

criteria = ['specificity', 'depth', 'actionability', 'objectivity', 'overall']
labels   = ['P1: Zero-Shot', 'P2: Chain-of-Thought', 'P3: Red-Team']
keys     = ['prompt_1', 'prompt_2', 'prompt_3']

score_matrix = np.array([
    [judge_scores[k][c] for c in criteria] for k in keys
])

df_scores = pd.DataFrame(
    score_matrix,
    index=labels,
    columns=[c.capitalize() for c in criteria]
)

print('\nJudge Score Summary:')
print(df_scores.to_string())
print(f"\n→ Best Prompt : {judge_scores['best_prompt'].replace('_', ' ').title()}")
print(f"→ Rationale   : {judge_scores['judge_rationale']}")

---
## Section 4 — Visualisations

Four figures are produced:
1. **Fig 1** — Grouped bar chart: average risk severity per category, Block vs PayPal
2. **Fig 2** — Scatter plot: Severity vs. Disclosure Explicitness (points in red zone = potential undisclosed risks)
3. **Fig 3** — Radar / spider chart: LLM-as-Judge scores across four criteria
4. **Fig 4** — Heatmap trilogy: Block risk scores, PayPal risk scores, and divergence (Block − PayPal)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 11 — Figure 1: Risk Severity by Category — Block vs PayPal
#
# A grouped bar chart showing the average severity score for each risk category,
# split by company. Higher bars = more severe risk disclosure.
# ─────────────────────────────────────────────────────────────────────────────

COLORS = {'Block': '#e05c3a', 'PayPal': '#003087'}
cat_order = df['Category'].unique()

# Compute per-category average severity
agg = (
    df.groupby(['Category', 'Company'])['Severity_Score']
    .mean()
    .unstack()
    .reindex(cat_order)
)

fig, ax = plt.subplots(figsize=(11, 5))
x     = np.arange(len(cat_order))
width = 0.35

for i, company in enumerate(['Block', 'PayPal']):
    vals = agg[company].values
    bars = ax.bar(
        x + i * width, vals, width,
        label=company, color=COLORS[company], alpha=0.85, edgecolor='white', linewidth=0.8
    )
    # Annotate each bar with its value
    for j, v in enumerate(vals):
        ax.text(
            x[j] + i * width, v + 0.07,
            f'{v:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold'
        )

ax.set_xticks(x + width / 2)
ax.set_xticklabels(
    [c.replace(' / ', '\n').replace(' & ', '\n') for c in cat_order],
    rotation=0, ha='center', fontsize=8.5
)
ax.set_ylabel('Average Severity Score (1–5)', fontsize=10)
ax.set_ylim(0, 6.2)
ax.set_title(
    'Figure 1 — Risk Severity by Category: Block vs PayPal (10-K 2023)',
    fontsize=12, fontweight='bold', pad=14
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('fig1_severity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Saved: fig1_severity_comparison.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 12 — Figure 2: Severity vs. Disclosure Explicitness Scatter
#
# Points in the upper-left quadrant (high severity, low explicitness) represent
# the "Undisclosed Risk Zone" — risks that are serious but poorly disclosed.
# These are the most concerning items from a regulatory / investor perspective.
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 7))

for company, grp in df.groupby('Company'):
    ax.scatter(
        grp['Disclosure_Explicitness'], grp['Severity_Score'],
        label=company, s=130, alpha=0.78,
        color=COLORS[company], edgecolors='white', linewidths=0.9, zorder=3
    )
    # Annotate each point with a short label
    for _, row in grp.iterrows():
        short = textwrap.shorten(row['Risk Factor'], width=38, placeholder='…')
        ax.annotate(
            short,
            (row['Disclosure_Explicitness'], row['Severity_Score']),
            fontsize=6.2, xytext=(5, 3), textcoords='offset points', alpha=0.72
        )

# Shade the undisclosed risk zone (high severity, low explicitness)
ax.axhspan(3.5, 5.5, xmin=0, xmax=0.42, alpha=0.07, color='red', zorder=1)
ax.text(
    1.08, 4.75,
    '⚠ HIGH SEVERITY\nLOW DISCLOSURE\n(Undisclosed Risk Zone)',
    fontsize=8, color='red', alpha=0.65, style='italic'
)

ax.set_xlabel('Disclosure Explicitness  (1 = vague  →  5 = quantified multi-paragraph)', fontsize=10)
ax.set_ylabel('Severity Score  (1 = minor  →  5 = primary risk)', fontsize=10)
ax.set_title(
    'Figure 2 — Severity vs. Disclosure Explicitness\n'
    '(Red zone = high severity but under-disclosed — potential hidden risk)',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3, linestyle='--')
ax.set_xlim(0.5, 5.8)
ax.set_ylim(1.5, 5.8)

plt.tight_layout()
plt.savefig('fig2_severity_vs_disclosure.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Saved: fig2_severity_vs_disclosure.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 13 — Figure 3: LLM-as-Judge Radar Chart
#
# A spider / radar chart visualises the four scoring dimensions simultaneously.
# Each coloured polygon represents one prompting technique.
# A larger polygon area = better overall analytical quality.
# ─────────────────────────────────────────────────────────────────────────────

radar_criteria   = ['Specificity', 'Depth', 'Actionability', 'Objectivity']
radar_keys_lower = ['specificity', 'depth', 'actionability', 'objectivity']
N      = len(radar_criteria)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

prompt_colors = ['#e05c3a', '#2196F3', '#4CAF50']

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_rlim(0, 10)
ax.set_rticks([2, 4, 6, 8, 10])
ax.set_rlabel_position(30)
ax.set_thetagrids(np.degrees(angles[:-1]), radar_criteria, fontsize=11)

for key, label, color in zip(keys, labels, prompt_colors):
    vals = [judge_scores[key][c] for c in radar_keys_lower]
    vals += vals[:1]  # close polygon
    ax.plot(angles, vals, color=color, linewidth=2.2, label=label)
    ax.fill(angles, vals, color=color, alpha=0.13)

ax.set_title(
    'Figure 3 — LLM-as-Judge: Prompt Quality Radar\n'
    '(scored by LLM on 4 criteria, max = 10)',
    fontsize=11, fontweight='bold', pad=22
)
ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.18), fontsize=9)

plt.tight_layout()
plt.savefig('fig3_judge_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Saved: fig3_judge_radar.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 14 — Figure 4: Risk Divergence Heatmap Trilogy
#
# Three side-by-side heatmaps:
#   4a  Block's risk scores across 5 dimensions
#   4b  PayPal's risk scores across the same 5 dimensions
#   4c  Divergence matrix (Block − PayPal): red = Block higher, green = PayPal higher
#
# Risk dimensions are analyst-assigned estimates based on 10-K filing language.
# ─────────────────────────────────────────────────────────────────────────────

risk_dims = [
    'Regulatory\nExposure',
    'Crypto\nConcentration',
    'Operational\nComplexity',
    'Revenue\nVisibility',
    'Strategic\nOverextension',
]

cat_labels = [
    'Regulatory', 'Cryptocurrency', 'Cybersecurity',
    'Technology', 'Financial', 'Operational'
]

# Analyst-estimated risk scores per category × dimension
block_risk_matrix = np.array([
    # Reg  Crypto  Ops  Rev  Strategic
    [5,    2,      4,   3,   3],   # Regulatory & Compliance
    [4,    5,      4,   2,   4],   # Cryptocurrency
    [3,    2,      5,   3,   3],   # Cybersecurity
    [2,    3,      4,   2,   5],   # Technology & Innovation
    [4,    4,      3,   4,   4],   # Financial & Market
    [3,    2,      4,   2,   5],   # Operational & Strategic
])

paypal_risk_matrix = np.array([
    [4,    1,      3,   4,   2],
    [3,    2,      3,   3,   2],
    [4,    1,      4,   4,   2],
    [2,    1,      3,   3,   2],
    [3,    1,      3,   4,   2],
    [3,    1,      4,   4,   2],
])

# Positive values = Block carries MORE risk than PayPal in that cell
divergence_matrix = block_risk_matrix - paypal_risk_matrix

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

plot_configs = [
    (block_risk_matrix,  'Block Risk Scores',           'Reds',     1,  5),
    (paypal_risk_matrix, 'PayPal Risk Scores',           'Blues',    1,  5),
    (divergence_matrix,  'Divergence (Block − PayPal)',  'RdYlGn_r', -3, 3),
]

for i, (ax, (matrix, title, cmap, vmin, vmax)) in enumerate(zip(axes, plot_configs)):
    sns.heatmap(
        matrix, ax=ax,
        xticklabels=risk_dims, yticklabels=cat_labels,
        annot=True, fmt='d', cmap=cmap,
        vmin=vmin, vmax=vmax,
        linewidths=0.5, linecolor='white',
        cbar_kws={'shrink': 0.8}
    )
    subfig_label = chr(ord('a') + i)
    ax.set_title(f'Figure 4{subfig_label} — {title}', fontsize=10, fontweight='bold', pad=8)
    ax.tick_params(axis='x', rotation=20, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle(
    'Figure 4 — Risk Divergence Heatmap: Block vs PayPal by Category & Dimension',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('fig4_divergence_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Saved: fig4_divergence_heatmap.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 15 — Figure 5: Judge Overall Score Bar Chart
#
# A simple horizontal bar chart making it immediately clear which prompting
# technique the judge rated highest on each criterion.
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 4))

x_pos    = np.arange(len(criteria))
bar_w    = 0.25
p_colors = ['#e05c3a', '#2196F3', '#4CAF50']

for i, (key, label, color) in enumerate(zip(keys, labels, p_colors)):
    vals = [judge_scores[key][c] for c in criteria]
    bars = ax.bar(x_pos + i * bar_w, vals, bar_w,
                  label=label, color=color, alpha=0.85, edgecolor='white')
    for j, v in enumerate(vals):
        ax.text(
            x_pos[j] + i * bar_w, v + 0.15,
            str(v), ha='center', va='bottom', fontsize=8.5, fontweight='bold'
        )

ax.set_xticks(x_pos + bar_w)
ax.set_xticklabels([c.capitalize() for c in criteria], fontsize=10)
ax.set_ylabel('Score (out of 10)', fontsize=10)
ax.set_ylim(0, 12)
ax.set_title(
    'Figure 5 — LLM-as-Judge Scores per Criterion & Prompting Technique',
    fontsize=11, fontweight='bold', pad=12
)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('fig5_judge_scores_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Saved: fig5_judge_scores_bar.png')

---
## Section 5 — Executive Summary Generation

The **winning prompt's output** (as identified by the judge) is fed into a final synthesis call
to produce a **200-word Executive Summary** answering the base question:

> *"Is Block's AI & Bitcoin focus creating undisclosed operational risks compared to PayPal's conservative approach?"*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 16 — Identify Winning Prompt
#
# The judge has already nominated a winner; we extract it here and retrieve
# the corresponding raw LLM response to use as input for the executive summary.
# ─────────────────────────────────────────────────────────────────────────────

best_key  = judge_scores['best_prompt']          # e.g. 'prompt_2'
best_map  = {'prompt_1': p1_response, 'prompt_2': p2_response, 'prompt_3': p3_response}
best_text = best_map[best_key]
best_label = labels[keys.index(best_key)]

print(f'Winning prompt      : {best_label}')
print(f'Judge overall score : {judge_scores[best_key]["overall"]}/10')
print(f'Judge comment       : {judge_scores[best_key]["comment"]}')
print(f'Judge rationale     : {judge_scores["judge_rationale"]}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 17 — Generate 200-Word Executive Summary
#
# The summary is written from the perspective of a CRO briefing the board.
# It synthesises the winning prompt's findings into flowing prose,
# covering the four required angles: divergence, top underweighted risks,
# PayPal's posture, and a final investment/risk verdict.
# ─────────────────────────────────────────────────────────────────────────────

EXEC_SYSTEM = """You are a Chief Risk Officer writing a brief for the board of directors.
Your summary must be exactly 200 words (±10 words), professional, and evidence-based.
Do not use bullet points — write in clean, flowing prose."""

EXEC_USER = f"""Based on the following risk analysis of Block, Inc. vs. PayPal Holdings (10-K 2023),
write a 200-word Executive Summary answering this question:

"Is Block's AI & Bitcoin focus creating undisclosed operational risks compared to PayPal's
 conservative, efficiency-first approach?"

=== BEST ANALYSIS (from {best_label}) ===
{best_text}

Your summary MUST cover:
1. The core risk divergence between Block and PayPal
2. The top two underweighted risks in Block's disclosure
3. Whether PayPal's conservative posture is genuinely lower-risk or merely better-disclosed
4. A final investment / risk management verdict
"""

print('Generating Executive Summary from winning prompt output...')
exec_summary = call_llm(EXEC_SYSTEM, EXEC_USER, max_tokens=450)

word_count = len(exec_summary.split())
print(f'\nWord count: {word_count}')
print('\n' + '═'*70)
print('  EXECUTIVE SUMMARY')
print('═'*70)
print(exec_summary)
print('═'*70)

---
## Section 6 — Summary Table & Final Findings

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 18 — Comparative Summary Table
# Side-by-side comparison of key metrics extracted from both 10-K filings.
# ─────────────────────────────────────────────────────────────────────────────

summary_df = pd.DataFrame({
    'Metric': [
        'Total Payment Volume / GPV (FY2022)',
        'Net Income / (Loss) FY2022',
        'Active Accounts / Transacting Users',
        'Bitcoin Balance Sheet Exposure',
        'Regulated Banking Subsidiary',
        'Number of Distinct Business Ecosystems',
        'BNPL Product Exposure',
        'Broker-Dealer Regulation',
        'Avg Risk Severity Score (6 categories)',
        'Avg Disclosure Explicitness Score',
    ],
    'Block, Inc.': [
        '$186.5B (Square GPV)',
        '($540.7M)',
        '51M Cash App MTAs',
        'Yes — direct balance sheet holdings',
        'Square Financial Services (FDIC-insured ILC)',
        '5  (Square, Cash App, TIDAL, TBD, Spiral)',
        'Yes (Afterpay / BNPL)',
        'Yes (Cash App Investing / FINRA)',
        f"{df[df.Company=='Block']['Severity_Score'].mean():.2f}",
        f"{df[df.Company=='Block']['Disclosure_Explicitness'].mean():.2f}",
    ],
    'PayPal Holdings': [
        '$1.36T (TPV)',
        'Profitable (efficiency focus)',
        '435M active accounts',
        'No direct holdings',
        'PayPal Europe (Luxembourg, ECB)',
        '2  (PayPal / Venmo core products)',
        'Pay Later (smaller scale)',
        'No (crypto via custodian only)',
        f"{df[df.Company=='PayPal']['Severity_Score'].mean():.2f}",
        f"{df[df.Company=='PayPal']['Disclosure_Explicitness'].mean():.2f}",
    ],
})

summary_df.set_index('Metric', inplace=True)

print('Block vs. PayPal — Key Comparative Metrics')
print('─' * 80)
print(summary_df.to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 19 — Final Judge Score Printout & Conclusion
# ─────────────────────────────────────────────────────────────────────────────

print('─' * 65)
print('  LLM-AS-JUDGE — FINAL SCORES')
print('─' * 65)

for key, label in zip(keys, labels):
    s = judge_scores[key]
    print(
        f'{label:28s}  '
        f'Overall: {s["overall"]}/10  |  '
        f'{s["comment"]}'
    )

print()
print(f'→ Winning prompt  : {best_label}')
print(f'→ Judge rationale : {judge_scores["judge_rationale"]}')
print('─' * 65)

---
## Key Findings

| Finding | Detail |
|---------|--------|
| **Bitcoin Concentration** | Block holds Bitcoin directly on its balance sheet — a unique risk with no PayPal equivalent. Impairment charges could amplify already-reported net losses beyond the $540.7M figure. |
| **Regulatory Stack Depth** | Block operates **5 regulated product lines** (Square, Cash App, TIDAL, TBD, Spiral) vs. PayPal's 2 core products. Each layer adds incremental regulatory, operational, and reputational exposure. |
| **BNPL Systemic Risk** | Afterpay integration exposes Block to BNPL regulatory crackdowns and credit-quality deterioration not yet fully quantified in disclosures. |
| **Industrial Bank (ILC) Risk** | Square Financial Services (FDIC-insured Utah ILC) creates deposit insurance obligations and Dodd-Frank systemic oversight — unique among pure-play fintechs. |
| **Ecosystem Opacity** | TIDAL, TBD, and Spiral have **no quantified revenue/loss projections** in the 10-K — material strategic risks with minimal disclosure explicitness. |
| **PayPal's Cyber Transparency** | PayPal leads on cybersecurity disclosure depth (TIO breach explicitly cited with 1.6M customer count); Block's equivalent disclosures are broader but less specific. |

---
*Analysis based on Block 10-K 2023 and PayPal 10-K 2023 (SEC EDGAR). All scores are analyst estimates based on filing language intensity and prominence.*  
*This notebook is for **educational and research purposes only** and does not constitute investment advice.*